In [1]:
"""
Models to try:

- LightGBM
"""

'\nModels to try:\n\n- LightGBM\n'

In [2]:
import pandas as pd


In [3]:
"""
Features:

 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          2111 non-null   object 
 1   Age                             2111 non-null   float64
 2   family_history_with_overweight  2111 non-null   object 
 3   FAVC                            2111 non-null   object 
 4   FCVC                            2111 non-null   float64
 5   NCP                             2111 non-null   float64
 6   CAEC                            2111 non-null   object 
 7   SMOKE                           2111 non-null   object 
 8   CH2O                            2111 non-null   float64
 9   SCC                             2111 non-null   object 
 10  FAF                             2111 non-null   float64
 11  TUE                             2111 non-null   float64
 12  CALC                            2111 non-null   object 
 13  MTRANS                          2111 non-null   object 
 14  NObeyesdad                      2111 non-null   object 
"""

# Keep only specified features and remove Unnamed and Height/Weight columns
COLUMNS_TO_KEEP = [
    'Gender', 'Age', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP',
    'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad'
]

NUMERICAL_FEATURES = ['Age'] #, 'Height', 'Weight']
BOOLEAN_FEATURES = ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
CATEGORICAL_FEATURES = ['CAEC', 'CALC', 'MTRANS', 'Gender']


In [4]:
# Load the datasets

raw_real_data = pd.read_csv("/work/datasets/real-data-20250501-154339.csv")
raw_synthetic_data = pd.read_csv("/work/datasets/synth_clean_20.csv")

RAW_DATA = {
    'real': raw_real_data,
    'synthetic': raw_synthetic_data
}

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# yes/no to 1/0
class BooleanToBinaryTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.replace({'yes': 1, 'no': 0})
        
# Define features
#numerical_features = ['Age', 'Height', 'Weight', 'BMI', "IBW"]


# Define transformers
numerical_transformer = Pipeline([
    ('scaler', StandardScaler())
])

boolean_transformer = Pipeline([
    ('boolean_to_binary', BooleanToBinaryTransformer())
])

categorical_transformer = Pipeline([
    ('onehotencoder',OneHotEncoder(drop='first', sparse_output=False))
])

# Combine all transformers into one preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, NUMERICAL_FEATURES),
        ('bool', boolean_transformer, BOOLEAN_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough' # Leave the rest of the columns as they are
)

In [6]:
def get_final_feature_names(preprocessor, X_df, numerical_features, boolean_features, categorical_features):
    """
    Reconstructs the full list of feature names after transformation.

    Parameters:
        preprocessor: the fitted ColumnTransformer
        X_df: the original unprocessed DataFrame (e.g., X_train)
        numerical_features, boolean_features, categorical_features: lists of assigned features

    Returns:
        A list of final feature names in the order they appear in the transformed array
    """

    #Assigned features
    assigned_features = numerical_features + boolean_features + categorical_features

    #Passthrough features (not transformed)
    all_features = list(X_df.columns)
    passthrough_features = [f for f in all_features if f not in assigned_features]

    #Get the feature names from OneHotEncoder
    cat_ohe = preprocessor.named_transformers_['cat']['onehotencoder']
    cat_feature_names = cat_ohe.get_feature_names_out(categorical_features)

    #Combine all
    final_feature_names = numerical_features + boolean_features + list(cat_feature_names) + passthrough_features

    return final_feature_names

In [11]:
from sklearn.model_selection import train_test_split

def preprocess(df):
    # Drop unnecessary columns
    df = df[COLUMNS_TO_KEEP]

    # Preprocess the data
    X = df.drop(columns=['NObeyesdad'])
    y = df['NObeyesdad']

    X = pd.DataFrame(preprocessor.fit_transform(X), columns=get_final_feature_names(preprocessor, X, NUMERICAL_FEATURES, BOOLEAN_FEATURES, CATEGORICAL_FEATURES))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }

In [14]:

PROCESSED_DATA = {}

for name, df in RAW_DATA.items():
    PROCESSED_DATA.update({name: preprocess(df)})

PROCESSED_DATA

{'real': {'X_train':            Age  family_history_with_overweight  FAVC  SMOKE  SCC  \
  162  -0.522124                             1.0   1.0    1.0  0.0   
  2001 -0.533953                             1.0   1.0    0.0  0.0   
  1435 -0.222692                             1.0   1.0    0.0  0.0   
  649  -0.390041                             0.0   0.0    0.0  0.0   
  1280  0.265136                             1.0   1.0    0.0  0.0   
  ...        ...                             ...   ...    ...  ...   
  1638  1.257405                             1.0   1.0    0.0  0.0   
  1095 -0.206889                             1.0   1.0    0.0  0.0   
  1130 -0.208489                             1.0   1.0    0.0  0.0   
  1294 -0.206889                             1.0   1.0    0.0  0.0   
  860  -0.679742                             1.0   1.0    0.0  0.0   
  
        CAEC_Frequently  CAEC_Sometimes  CAEC_no  CALC_Frequently  \
  162               0.0             0.0      0.0              0.0   


In [27]:
# Apply preprocessing pipeline


'\nFeatures:\n\n #   Column                          Non-Null Count  Dtype  \n---  ------                          --------------  -----  \n 0   Gender                          2111 non-null   object \n 1   Age                             2111 non-null   float64\n 2   family_history_with_overweight  2111 non-null   object \n 3   FAVC                            2111 non-null   object \n 4   FCVC                            2111 non-null   float64\n 5   NCP                             2111 non-null   float64\n 6   CAEC                            2111 non-null   object \n 7   SMOKE                           2111 non-null   object \n 8   CH2O                            2111 non-null   float64\n 9   SCC                             2111 non-null   object \n 10  FAF                             2111 non-null   float64\n 11  TUE                             2111 non-null   float64\n 12  CALC                            2111 non-null   object \n 13  MTRANS                          2111 non-null   o

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=a441f35e-4b4c-4c50-b56a-1aea6b800ed8' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>